# Redrob Ranker Sandbox Demo

This notebook verifies that the repository can install dependencies, fetch Git LFS assets, prepare the local reranker weights, run the ranking step, patch the audited reasoning rows, validate the CSV, and preview the output.

Repository: `https://github.com/Prudhvi-3-cloud/Redrob-AI.git`


In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/Prudhvi-3-cloud/Redrob-AI.git"
PROJECT_DIR = Path("/content/redrob-ranker")

!git lfs version || (apt-get update -qq && apt-get install -y git-lfs)
!git lfs install

if PROJECT_DIR.exists():
    print(f"Using existing directory: {PROJECT_DIR}")
else:
    !git clone "$REPO_URL" "$PROJECT_DIR"

print("PROJECT_DIR =", PROJECT_DIR)


In [ ]:
%cd $PROJECT_DIR
!git lfs pull
!python --version
!python -m pip install -q -r requirements.txt


In [ ]:
from pathlib import Path

reranker_weight = Path("models/bge_reranker/model.safetensors")
if not reranker_weight.exists():
    print("Downloading local reranker weights for sandbox setup...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        "BAAI/bge-reranker-v2-m3",
        local_dir="models/bge_reranker",
        ignore_patterns=[".git*", "assets/*", "*.gguf", "onnx/*"],
    )
else:
    print("Reranker weights already present.")


In [ ]:
from pathlib import Path

required = [
    "rank.py",
    "patch_reasoning.py",
    "validate_submission.py",
    "requirements.txt",
    "candidates.jsonl",
    "job_description.docx",
    "artifacts/candidate_embeddings.npy",
    "artifacts/candidate_ids.json",
    "artifacts/candidates_parsed.pkl",
    "artifacts/bm25s_index",
    "models/bge_small/config.json",
    "models/bge_small/model.safetensors",
    "models/bge_reranker/config.json",
    "models/bge_reranker/model.safetensors",
]

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing))
print("Required files found.")


In [ ]:
!python rank.py --candidates ./candidates.jsonl --job-description ./job_description.docx --cross-encoder-limit 30 --out ./submission.csv
!python patch_reasoning.py
!python validate_submission.py --submission ./submission.csv

In [ ]:
import csv

with open("submission.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print("rows:", len(rows))
for row in rows[:5]:
    print(row["rank"], row["candidate_id"], row["score"], row["reasoning"][:120])